# DRLCA on Colab — OCR the Disability Act + foundation pipeline

Heavy work lives here (free Colab GPU/CPU + Gemini free tier). Local `ds-general` is for light editing/verification only.

**Setup:** Runtime → Change runtime type → T4 GPU optional (CPU is fine). Add `GOOGLE_API_KEY` under 🔑 Secrets (name exactly `GOOGLE_API_KEY`).

In [ ]:
%pip install -q pymupdf google-generativeai 2>&1 | tail -2

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = API_KEY
ROOT = '/content/drive/MyDrive/DRLCA'
os.makedirs(f'{ROOT}/raw', exist_ok=True)
os.makedirs(f'{ROOT}/processed', exist_ok=True)
print('ROOT:', ROOT)

In [ ]:
import requests
pdf_url = 'https://qualitativemagazine.com/wp-content/uploads/2022/10/1244-Discrimination-Against-Persons-with-Disabilities-Prohibition-ACT-2018.pdf'
pdf_path = f'{ROOT}/raw/disability_act_2018_full.pdf'
if not os.path.exists(pdf_path):
    r = requests.get(pdf_url, headers={'User-Agent': 'Mozilla/5.0 DRLCA-Colab'}, timeout=180)
    r.raise_for_status()
    open(pdf_path, 'wb').write(r.content)
print(f'{os.path.getsize(pdf_path) / 1e6:.1f} MB:', pdf_path)

In [ ]:
import json, time
import pymupdf
import google.generativeai as genai

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('models/gemini-2.5-flash')
PROMPT = ('Transcribe this scanned legal document page exactly, preserving section numbers, '
           'headings, and paragraph structure. Return only the transcribed text, no commentary. '
           'If a region is illegible, write [illegible].')
CKPT = f'{ROOT}/processed/ocr_checkpoint.json'
ckpt = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
doc = pymupdf.open(pdf_path)
print(f'pages: {len(doc)}, done: {len(ckpt)}')
for i in range(1, len(doc) + 1):
    if str(i) in ckpt:
        continue
    img = doc.load_page(i - 1).get_pixmap(dpi=200).tobytes('png')
    for attempt in range(5):
        try:
            ckpt[str(i)] = model.generate_content([PROMPT, {'mime_type': 'image/png', 'data': img}]).text.strip()
            json.dump(ckpt, open(CKPT, 'w'))
            print(f'page {i}/{len(doc)}: {len(ckpt[str(i)])} chars')
            break
        except Exception as e:
            print(f'page {i} retry {attempt}: {repr(e)[:120]}')
            time.sleep(30)  # free-tier quota: wait and resume; checkpoint keeps progress
    time.sleep(5)

In [ ]:
import re
missing = [i for i in range(1, len(doc) + 1) if str(i) not in ckpt]
assert not missing, f're-run the OCR cell — still missing: {missing}'
full = '\n'.join(f'===== PAGE {i} =====\n{ckpt[str(i)]}' for i in range(1, len(doc) + 1))
full = re.sub(r'[^\S\n]+', ' ', full).strip()
out = f'{ROOT}/processed/disability_act_2018_full.txt'
open(out, 'w', encoding='utf-8').write(full)
print(f'{len(full)} chars -> {out}')
print('Sections found:', len(re.findall(r'Section \d+', full, re.I)))